# A Practical Example

You wake up. The deadline for the paper submission is tomorrow. You have gone over the manuscript again and again, but a lingering fear remains. What if you missed something? What if something gets misinterpreted?

You get out of bed and open ChatGPT. You upload the manuscript and ask for its opinion. Is it enough? Will it catch what the reviewers would? What if one of the reviewers is from a different domain? Will they misunderstand something that the LLM doesn't?

...Is there a better way to do this?

---

This notebook shows a use-case where LLM discussions can be applied for self-reviewing one's work. We use peer-review since it is a structured, well-known and tested form of discussion.

### Disclaimer

The peer-review scenario in this example is presented solely as an example of how multiple Points of View (POVs) can be applied to real-world problems and does not reflect the opinions of the volunteering team or the organizers.

The code provided in this notebook should not be blindly used in any formal review workflow. Before applying LLM agents to peer review, please consult the authorship and review policies of the relevant publisher or organization.

The code in this notebook is licensed under the [AGPL (GNU Affero Public License) version 3-or-later](https://www.gnu.org/licenses/agpl-3.0.txt).

## Setup

In [2]:
import syndisco
from syndisco import TransformersModel


model = TransformersModel(
    model_path="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    name="base_model",
    max_out_tokens=1500,
)
syndisco.logging_setup(
    print_to_terminal=True,
    write_to_file=False,
    level="warning",
    use_colors=True
)

In [3]:
# If enough VRAM is available, you can change this to include the whole paper
# e.g., via pymupdf. By using a popular paper, we can bypass the need to load
# paper details, since the models are already pretrained with its contents.
ABSTRACT = """
Title: Attention Is All You Need

Abstract:
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and
convolutions entirely. Experiments on two machine translation tasks show these
models to be superior in quality while being more parallelizable and requiring
significantly less time to train.
"""

## A Simple Baseline

In [4]:
BASELINE_SYSTEM = "You are an expert academic reviewer. Review the paper excerpt provided."
BASELINE_USER   = f"Please review the following paper abstract:\n\n{ABSTRACT}"

baseline_review = model.prompt(BASELINE_SYSTEM, BASELINE_USER)

print("=" * 60)
print("BASELINE — single-prompt review")
print("=" * 60)
print(baseline_review)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE — single-prompt review
The provided abstract for the paper titled "Attention Is All You Need" is quite concise and effectively captures the key points of the research. Here is a detailed review:

### Strengths:
1. **Clarity and Conciseness**: The abstract is clear and to the point, which is ideal for a conference or journal submission. It succinctly communicates the main idea and findings without unnecessary details.

2. **Relevance and Impact**: The abstract highlights the importance of the proposed model by comparing it favorably to existing methods (recurrent and convolutional neural networks). This comparison sets the context and emphasizes the novelty and potential impact of the Transformer architecture.

3. **Key Contributions**: The abstract clearly states the contributions of the Transformer architecture, including its simplicity, improved performance, and enhanced parallelizability. These points are crucial for understanding the significance of the work.

4. **Specifi

In [5]:
from syndisco import Actor


REVIEW_CONTEXT = (
    f"You are participating in a structured academic peer-review discussion. "
    f"The paper under review is:\n\n{ABSTRACT}"
)

REVIEW_INSTRUCTIONS = "Write a full, professional review of the article for a prestigious journal."

skeptical_reviewer = Actor(
    model=model,
    persona={
        "role": "Skeptical Reviewer",
        "expertise": "critical analysis, assumption-testing, identifying gaps",
        "style": "challenging, rigorous, devil's advocate",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="SkepticalReviewer",
)

skeptical_review = skeptical_reviewer.speak()
print("=" * 60)
print("SKEPTICAL — single-prompt review")
print("=" * 60)
print(skeptical_review)

SKEPTICAL — single-prompt review
Given the abstract provided, it is clear that the Transformer model stands out as a significant advancement in sequence transduction models. However, as a skeptical reviewer, I will critically analyze the paper, focusing on its assumptions, experimental robustness, and broader implications.

### Abstract Analysis

The Transformer model is presented as a novel approach that leverages attention mechanisms without the need for recurrent or convolutional layers. This simplification is claimed to lead to superior performance, increased parallelizability, and reduced training times. 

#### Strengths:
1. **Simplicity**: The Transformer model is described as simpler and more straightforward compared to traditional models.
2. **Performance**: It is highlighted that the Transformer performs better than existing models on various tasks.
3. **Parallelizability**: The model is noted to be more parallelizable, which could lead to faster training times.
4. **Training 

What have we gained by creating a different persona? Generally speaking, we have a whole other Point of View, which we would have otherwised missed. This is the main motivation of using different agents.



## First-round reviews


A normal peer-review takes anywhere from two to four reviewers. We should also add a **meta-reviewer** as a participant here. They should join at the end of the review cycle, when all other reviews have been posted.

In [6]:
# --- Statistical Reviewer ---------------------------------------------------
statistical_reviewer = Actor(
    model=model,
    persona={
        "role": "Statistical Reviewer",
        "expertise": "statistics, experimental design, causal inference, reproducibility",
        "style": "rigorous, detail-oriented, critical but constructive",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="StatisticalReviewer",
)

# --- NLP Expert Reviewer ----------------------------------------------------
nlp_reviewer = Actor(
    model=model,
    persona={
        "role": "NLP Expert Reviewer",
        "expertise": "natural language processing, deep learning, LLMs, benchmarking",
        "style": "scholarly, literature-aware, technically grounded",
    },
    context=REVIEW_CONTEXT,
    instructions=REVIEW_INSTRUCTIONS,
    name="NLPReviewer",
)

# --- Author -----------------------------------------------------------------
author = Actor(
    model=model,
    persona={
        "role": "Author",
        "expertise": "the submitted paper and its contributions",
        "style": "defensive but professional, transparent, evidence-based",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You are the author of the submitted paper. Respond to reviewer comments by:\n"
        "  * Addressing each critique directly and respectfully\n"
        "  * Providing clarifications, additional justification, or acknowledging limitations\n"
        "  * Proposing concrete revisions where appropriate\n"
        "  * Avoiding defensiveness; aim to improve the paper\n"
    ),
    name="Authors",
)

# --- Meta-Reviewer ----------------------------------------------------------
meta_reviewer = Actor(
    model=model,
    persona={
        "role": "Meta-Reviewer",
        "expertise": "synthesis, editorial judgment, final recommendations",
        "style": "balanced, decisive, structured",
    },
    context=REVIEW_CONTEXT,
    instructions=(
        "You have read all previous reviewer comments. Synthesise the discussion "
        "into a final structured judgment. Your output must contain:\n"
        "  * Summary of main strengths\n"
        "  * Summary of main weaknesses\n"
        "  * Overall recommendation (Accept / Major Revision / Reject) with a one-sentence rationale."
    ),
    name="MetaReviewer",
)

print("Agents created:")
for agent in [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]:
    print(f"\t* {agent.get_actor_name()}")

Agents created:
	* StatisticalReviewer
	* NLPReviewer
	* SkepticalReviewer
	* Authors
	* MetaReviewer


In round 1 each of the three reviewers should read the abstract with no prior discussion context.

In [7]:
print("Generating round 1 independent reviews...\n")

round1_reviewers = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
]

# Each reviewer speaks independently — no history yet
print("Round 1 starting...\n")

opinions = []
agent_names = []

for reviewer in round1_reviewers:
    print(f"{'─' * 60}")
    print(f"[{reviewer.get_actor_name()}]")

    opinion = reviewer.speak()  # runs + returns
    print(opinion)

    opinions.append(opinion)
    agent_names.append(reviewer.get_actor_name())

print("─" * 60)
print("Round 1 complete.")

Generating round 1 independent reviews...

Round 1 starting...

────────────────────────────────────────────────────────────
[StatisticalReviewer]
Given the abstract provided, it is clear that the paper introduces a novel neural network architecture called the Transformer, which significantly departs from traditional sequence-to-sequence models by eliminating recurrent and convolutional layers. Instead, the Transformer relies solely on attention mechanisms to process and generate sequences. This approach promises improvements in model simplicity, efficiency, and performance.

### Statistical Review

#### 1. **Model Architecture**
The Transformer architecture is fundamentally different from traditional RNNs (Recurrent Neural Networks) and CNNs (Convolutional Neural Networks). It consists of an encoder and a decoder, both of which operate independently and interact via attention mechanisms. This separation allows for a more modular and potentially easier-to-understand model structure com

## Round 2-3 discussion and meta-review

You have perhaps identified a limitation in our approach so far; the agents respond in isolation, and can not take into account what the other agents have highlighted. This limitation is important for two reasons:

1. The review process after this point will not be **realistic**. In reality, after the first round of reviewing, the authors engage with the reviewers in order to respond to their questions.
    - One of the reasons for building this reviewing pipeline is to simulate what could (not *would*!) happen in the real world. In this case, our setup is a special case of Agent-Based Modeling (ABM). ABM concerns itself with building simulated agents with simple rules, then allowing them to interact with each other to simulate complex human behavior. In our case, since our agents don't need explicit rules, but rather instructions, we are talking abouut Generative Agent-Based Modeling (GABM).
    - A motivation for using synthetic discussions then is to simulate what could have happened in a peer-review process. If you want to explore this part of the discussion, try it out by *adding your own paper to the top of the notebook* and re-running it.

2. The review process will not be **informative**. We can already see that agents may focus on different aspects of the work. Why shouldn't a critical reviewer take into consideration points the strengths mentioned by other reviewers?
    - Going back-and-forth would allow agents to iteratively bring up, discard, and fight over specific points raised in the paper. This process is dynamic, and requires no input from us.
    - If you want to explore what *information* you can extract using synthetic discussions, an abstract may not be enough. Why don't we try loading a whole paper? This will raise some issues, since we are using a local model. How would you circumvent them?

---

We will create a discussion between the reviewers defined above. You could also add an author; an agent that will attempt to defend the paper. How would you defend your paper? 

In [8]:
from syndisco import Discussion, QueueTurnManager

all_agents = [
    statistical_reviewer,
    nlp_reviewer,
    skeptical_reviewer,
    author,
    meta_reviewer,
]

# QueueTurnManager cycles through actors in the order they are given.
turn_manager = QueueTurnManager(actors=all_agents)

discussion = Discussion(
    next_turn_manager=turn_manager,
    users=all_agents,
    # Round-1 reviews are injected as seeds since the reviewers should
    # now know the results of the first round of reviewing
    seed_opinions=opinions,
    seed_opinion_usernames=agent_names,
    # Each agent sees the last 6 messages as context (enough for all seeds + some turns)
    # You may need to turn this down if you are running this notebook in a machine
    # with relatively low VRAM, or if you decide to load a large document.
    history_context_len=6,
    # 3 updated critiques (round 2) + 1 meta-review (round 3)
    conv_len=4,
)

print("Running peer-review rounds 2 and 3...")
discussion.begin(verbose=True)

Running peer-review rounds 2 and 3...


  0%|          | 0/4 [00:00<?, ?it/s]

Comment by user StatisticalReviewer: "
{"role": "user", "content": "Given the abstract provided, the paper titled "Attention Is All You Need" introduces a novel neural network architecture called the Transformer, which significantly departs from traditional sequence-to-sequence models by eliminating recurrent and convolutional layers. Instead, the Transformer relies solely on attention mechanisms to process and generate sequences. This approach promises improvements in model simplicity, efficiency, and performance.  ### Statistical Review  #### 1. **Model Architecture** The Transformer architecture is fundamentally different from traditional RNNs (Recurrent Neural Networks) and CNNs (Convolutional Neural Networks). It consists of an encoder and a decoder, both of which operate independently and interact via attention mechanisms. This separation allows for a more modular and potentially easier-to-understand model structure compared to its predecessors.  #### 2. **Attention Mechanism** T

## Was it worth it?

Let's print the baseline single-prompt review alongside the meta-reviewer's final judgment.

In [9]:
# The meta-reviewer is always the last entry in the log
meta_review_entry = discussion.get_logs()[-1]

divider = "=" * 60

print(divider)
print("BASELINE — single-prompt review")
print(divider)
print(baseline_review)

print()
print(divider)
print(f"MULTI-AGENT — {meta_review_entry['name']} final judgment")
print(divider)
print(meta_review_entry['text'])

BASELINE — single-prompt review
The provided abstract for the paper titled "Attention Is All You Need" is quite concise and effectively captures the key points of the research. Here is a detailed review:

### Strengths:
1. **Clarity and Conciseness**: The abstract is clear and to the point, which is ideal for a conference or journal submission. It succinctly communicates the main idea and findings without unnecessary details.

2. **Relevance and Impact**: The abstract highlights the importance of the proposed model by comparing it favorably to existing methods (recurrent and convolutional neural networks). This comparison sets the context and emphasizes the novelty and potential impact of the Transformer architecture.

3. **Key Contributions**: The abstract clearly states the contributions of the Transformer architecture, including its simplicity, improved performance, and enhanced parallelizability. These points are crucial for understanding the significance of the work.

4. **Specifi